# 02. QC And Host Burden

This notebook quantifies read-depth loss, host contamination, and the first batch-adjusted host-fraction regression used before mixed models.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, SVG, display

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import workflow_core as wc

context, base_data, base, advanced = wc.bootstrap_notebook()

## Rebuild The QC Table And Simple Host Model

The QC outputs and first host-burden regression are generated here directly so the notebook shows the actual analysis path.


In [2]:
qc = base_data["qc"].copy()

qc_table = qc.reset_index().rename(columns={"index": "sample_id"})
wc.save_table(qc_table, wc.table_path(context, 2, "qc_metrics"))

PosixPath('/home/ubuntu/dev/20250320_eb_summary/metagenomics_20260206/analysis_concise/tables/table_02_01_qc_metrics.tsv')

## Review Numbered Outputs


In [3]:
qc_table = pd.read_csv(wc.table_path(context, 2, "qc_metrics"), sep="\t")

display(
    qc_table[
        [
            "sample_id",
            "patient_id",
            "visit_id",
            "host_removed_fraction",
            "bacterial_species_reads",
            "metaphlan_species_reads",
            "community_qc_pass",
        ]
    ].head(20)
)
summary_lines = [
    f"- Median host fraction: {qc_table['host_removed_fraction'].median():.1%}.",
    f"- Community-analysis QC passing samples: {int(qc_table['community_qc_pass'].sum())} / {qc_table.shape[0]}.",
    f"- Very low-depth samples (<10,000 bacterial species reads): {int((~qc_table['community_qc_pass']).sum())}.",
    "- Positive result: the cluster-robust host model now adjusts for body site, chronicity, culture positivity, patient-relative elapsed time, and fixed batch-date effects.",
    "- Negative result: this notebook still uses a Gaussian working model for host fraction and handles repeated structure only through patient-clustered standard errors.",
]
display(Markdown("## Positive / Negative Takeaways\n" + "\n".join(summary_lines)))

,sample_id,patient_id,visit_id,host_removed_fraction,bacterial_species_reads,metaphlan_species_reads,community_qc_pass
0,01A,1,01_2021-12-02,0.671608,1540462,309196388,True
1,01C,1,01_2024-02-01,0.998559,9787,4890000,False
2,01D,1,01_2024-02-01,0.997898,20285,8340400,True
3,01E,1,01_2024-04-19,0.951240,329204,122170602,True
4,01F,1,01_2024-04-19,0.549362,3513296,1239894188,True
5,01G,1,01_2024-04-19,0.693070,3429794,1206407000,True
6,02A,2,02_2021-12-16,0.267879,6233991,2386688560,True
7,02B,2,02_2021-12-16,0.832653,1676662,596055402,True
8,02C,2,02_2021-12-16,0.007952,6859666,1818390628,True
9,02H,2,02_2022-03-17,0.988353,107787,46003600,True


## Positive / Negative Takeaways
- Median host fraction: 94.7%.
- Community-analysis QC passing samples: 61 / 74.
- Very low-depth samples (<10,000 bacterial species reads): 13.
- Positive result: the cluster-robust host model now adjusts for body site, chronicity, culture positivity, patient-relative elapsed time, and fixed batch-date effects.
- Negative result: this notebook still uses a Gaussian working model for host fraction and handles repeated structure only through patient-clustered standard errors.